In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# feature build fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. This notebook builds cross-row features (lags/rolling), which
# can't be row-batched, so the guard is the protection here.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 9.4G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298148864

In [2]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

REGION        = variables.TARGET_REGION
ALL_REGIONS   = ["nsw", "qld", "vic", "sa"]
OTHER_REGIONS = [r for r in ALL_REGIONS if r != REGION]

# Gas prices (STTM / DWGM) settle daily and are forward-filled to 5-min, so
# only calendar-day-scale lags/rolls carry information. The gas-day price is
# published ex-ante (day-ahead), so the contemporaneous value is leakage-free.
PER_HOUR = 60 // variables.FEATURE_GRANULARITY_IN_MINUTES   # 12
PER_DAY  = 24 * PER_HOUR                                     # 288
PER_WEEK = 7 * PER_DAY                                       # 2016

In [3]:
df = read_parquet_float32("../1_Dataset/Processed_data/4_STTM_DWGM.parquet")

df_core_columns = df.columns
df_base = df
df_base[:10]

Loading..: 100%|██████████| 9/9 [00:00<00:00, 129.33batch/s]


,gas_price_nsw,gas_price_sa,gas_price_qld,gas_price_vic
SETTLEMENTDATE,,,,
2018-01-01 00:00:00,5.5577,6.5055,5.6868,4.5
2018-01-01 00:05:00,5.5577,6.5055,5.6868,4.5
2018-01-01 00:10:00,5.5577,6.5055,5.6868,4.5
2018-01-01 00:15:00,5.5577,6.5055,5.6868,4.5
2018-01-01 00:20:00,5.5577,6.5055,5.6868,4.5
2018-01-01 00:25:00,5.5577,6.5055,5.6868,4.5
2018-01-01 00:30:00,5.5577,6.5055,5.6868,4.5
2018-01-01 00:35:00,5.5577,6.5055,5.6868,4.5
2018-01-01 00:40:00,5.5577,6.5055,5.6868,4.5


In [4]:
def _add_gas_level_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Contemporaneous gas-price levels and cross-region spreads. Gas is the
    marginal fuel that sets the electricity price cap in tight periods, so the
    gas price and its spread to neighbours are direct cost drivers. Gas-day
    prices are published ex-ante, hence leakage-free.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    gcols = [f"gas_price_{r}" for r in ALL_REGIONS if f"gas_price_{r}" in df.columns]
    nat = df[gcols].mean(axis=1)

    new_cols = {}
    new_cols[f"gas_price_{R}_now"]           = df[f"gas_price_{R}"].astype(np.float32)
    new_cols["gas_price_nat_avg"]            = nat.astype(np.float32)
    new_cols[f"gas_price_spread_{R}_vs_nat"] = (df[f"gas_price_{R}"] - nat).astype(np.float32)

    for r in OTHER_REGIONS:
        if f"gas_price_{r}" in df.columns:
            new_cols[f"gas_price_{r}_now"]           = df[f"gas_price_{r}"].astype(np.float32)
            new_cols[f"gas_price_spread_{R}_vs_{r}"] = (df[f"gas_price_{R}"] - df[f"gas_price_{r}"]).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_gas_level_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,gas_price_nsw_now,gas_price_nat_avg,gas_price_spread_nsw_vs_nat,gas_price_qld_now,gas_price_spread_nsw_vs_qld,gas_price_vic_now,gas_price_spread_nsw_vs_vic,gas_price_sa_now,gas_price_spread_nsw_vs_sa
SETTLEMENTDATE,,,,,,,,,
2018-01-01 00:00:00,5.5577,5.5625,-0.0048,5.6868,-0.1291,4.5,1.0577,6.5055,-0.9478
2018-01-01 00:05:00,5.5577,5.5625,-0.0048,5.6868,-0.1291,4.5,1.0577,6.5055,-0.9478
2018-01-01 00:10:00,5.5577,5.5625,-0.0048,5.6868,-0.1291,4.5,1.0577,6.5055,-0.9478
2018-01-01 00:15:00,5.5577,5.5625,-0.0048,5.6868,-0.1291,4.5,1.0577,6.5055,-0.9478
2018-01-01 00:20:00,5.5577,5.5625,-0.0048,5.6868,-0.1291,4.5,1.0577,6.5055,-0.9478
2018-01-01 00:25:00,5.5577,5.5625,-0.0048,5.6868,-0.1291,4.5,1.0577,6.5055,-0.9478
2018-01-01 00:30:00,5.5577,5.5625,-0.0048,5.6868,-0.1291,4.5,1.0577,6.5055,-0.9478
2018-01-01 00:35:00,5.5577,5.5625,-0.0048,5.6868,-0.1291,4.5,1.0577,6.5055,-0.9478
2018-01-01 00:40:00,5.5577,5.5625,-0.0048,5.6868,-0.1291,4.5,1.0577,6.5055,-0.9478


In [5]:
def _add_gas_dynamics_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Backward-looking gas-price dynamics for the target region: day/week lags,
    weekly/monthly rolling level and volatility, weekly momentum and a 90-day
    percentile rank (how expensive gas is relative to the recent regime).
    Look-back only, no leakage.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    s = df[f"gas_price_{R}"]
    new_cols = {}

    for lag in [PER_DAY, 2 * PER_DAY, PER_WEEK, 4 * PER_WEEK]:
        new_cols[f"gas_price_{R}_lag_{lag}"] = s.shift(lag).astype(np.float32)

    new_cols[f"gas_price_{R}_rmean_{PER_WEEK}"]   = s.rolling(PER_WEEK, min_periods=PER_DAY).mean().astype(np.float32)
    new_cols[f"gas_price_{R}_rstd_{4 * PER_WEEK}"] = s.rolling(4 * PER_WEEK, min_periods=PER_WEEK).std().astype(np.float32)
    new_cols[f"gas_price_{R}_mom_1w"]             = s.diff(PER_WEEK).astype(np.float32)
    new_cols[f"gas_price_{R}_pct_rank_90d"]       = (
        s.rolling(90 * PER_DAY, min_periods=7 * PER_DAY).rank(pct=True).astype(np.float32)
    )

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_gas_dynamics_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,gas_price_nsw_lag_288,gas_price_nsw_lag_576,gas_price_nsw_lag_2016,gas_price_nsw_lag_8064,gas_price_nsw_rmean_2016,gas_price_nsw_rstd_8064,gas_price_nsw_mom_1w,gas_price_nsw_pct_rank_90d
SETTLEMENTDATE,,,,,,,,
2018-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Retain core columns: gas-day prices are published ex-ante (known at t)
# -> leakage-free (already exposed via the *_now gas-level features).
print("Total features:", df.shape[1])
df.to_parquet("../2_Features_build/Feature_data/4_STTM_DWGM.parquet")
df.shape

Total features: 21


(893665, 21)

In [7]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 15 variable(s); kernel rss 0.24G, 9.2G RAM free now
